# Day 4: PHI Governance & De-Identification Engine

This notebook demonstrates the de-identification module applying Format-Preserving Encryption (FPE) for SSN/MRN, deterministic date-shifting (+/- 30 days), and free-text Safe Harbor 18 redaction.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ -q

In [ ]:
import os
import hashlib
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col, date_add, to_date
from pyspark.sql.types import StringType, DateType, IntegerType
import re

# Must set salt for FPE compliance
os.environ["EHDIP_HASH_SALT"] = "super_secret_enterprise_salt_123!"

spark = SparkSession.builder \
    .appName("Day04_PHI_DeID") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
def get_hash_salt():
    salt = os.environ.get("EHDIP_HASH_SALT")
    if not salt:
        raise ValueError("FATAL: HIPAA Compliance Error - EHDIP_HASH_SALT missing.")
    return salt

def fpe_mask(value):
    if not value:
        return value
    salt = get_hash_salt()
    return hashlib.sha256((str(value) + salt).encode('utf-8')).hexdigest()[:16]

def calculate_date_shift(patient_id):
    if not patient_id:
        return 0
    salt = get_hash_salt()
    h = hashlib.sha256((str(patient_id) + salt).encode('utf-8')).hexdigest()
    shift = (int(h[:8], 16) % 61) - 30
    return shift

def safe_harbor_redact(text):
    if not text:
        return text
    return re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[REDACTED SSN]', text)

fpe_mask_udf = udf(fpe_mask, StringType())
date_shift_udf = udf(calculate_date_shift, IntegerType())
safe_harbor_udf = udf(safe_harbor_redact, StringType())

In [ ]:
# Synthetic Data
data = [
    ("PAT001", "123-45-6789", "MRN999", "1980-05-15", "Patient feels fine. SSN is 123-45-6789."),
    ("PAT002", "987-65-4321", "MRN888", "1992-11-20", "No issues reported.")
]

df = spark.createDataFrame(data, ["patient_id", "ssn", "mrn", "dob", "notes"])
df = df.withColumn("dob", to_date(col("dob")))

print("RAW DATA (Contains PHI):")
df.show(truncate=False)

In [ ]:
# Apply De-ID
df_with_shift = df.withColumn("date_shift_days", date_shift_udf(col("patient_id")))

deid_df = df_with_shift \
    .withColumn("ssn_masked", fpe_mask_udf(col("ssn"))) \
    .withColumn("mrn_masked", fpe_mask_udf(col("mrn"))) \
    .withColumn("dob_shifted", date_add(col("dob"), col("date_shift_days"))) \
    .withColumn("notes_redacted", safe_harbor_udf(col("notes"))) \
    .drop("ssn", "mrn", "dob", "notes")

print("DE-IDENTIFIED DATA (Safe for Silver/Gold Analytics):")
deid_df.show(truncate=False)